# Western Balkan Studies

## Define RUN TAG

In [ ]:
RUN_ID = 'vre_low_we2023_20260324'
config_name='config_WB6_2023.yaml'

* load packages

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler

plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)


- Load Configs

In [ ]:
cfg=utils.load_config(f'../config/{config_name}')
# run_id:str=cfg.get('Scenario').get('run_id')

sub_national_unit_tag:str=cfg.get('GADM').get('datafield_mapping').get('NAME_2',None) or cfg.get('GADM').get('datafield_mapping').get('NAME_1')
country_name:str=cfg.get('country')
country_kwd=country_name.replace(' ','')
CRS_m = cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
regions=list(cfg.get('region_mapping').keys()) #'AL','BA','XK','ME','MK','RS'
vis_save_to_root=utils.ensure_path(f"../vis/{country_kwd}/{RUN_ID}/Combined_regions")

results_save_to_root=utils.ensure_path(f"../results/{country_kwd}/{RUN_ID}/Combined_regions")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
combined_store:dict[dict] = {}
utils.print_update(level=1,message=f"Loading data stores for {country_name} regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(f"../data/store/{country_kwd}/{RUN_ID}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        country_dict['timeseries_solar'] = res_data.from_store('timeseries/solar')
        country_dict['timeseries_wind'] = res_data.from_store('timeseries/wind')
        # country_dict['di_solar'] = res_data.from_store('dissolved_indices/solar')
        # country_dict['di_wind'] = res_data.from_store('dissolved_indices/wind')                                                                 
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        combined_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for : {cfg.get('region_mapping').get(region).get('name') if cfg.get('region_mapping').get(region) else region}.") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_dict:dict[pd.DataFrame]=[combined_store[region]['cells'] for region in combined_store]
all_cells_gdf = gpd.GeoDataFrame(pd.concat(all_cells_dict, ignore_index=False), crs=all_cells_dict[0].crs)

- Create cells' instance for plotting (CRS-m)

In [ ]:
if all_cells_gdf.crs != CRS_m:
    all_cells_gdf_plot = all_cells_gdf.to_crs(CRS_m)
else:
    all_cells_gdf_plot = all_cells_gdf

---

# Load Test/Validation data

In [ ]:
existing_VREs_data_path=Path("../data/validation_data/existing_VREs_WB6.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.Longitude,existing_VREs.Latitude),crs="EPSG:4326")
    utils.print_update(level=1,message=f"✓ Loaded validation data for existing VREs from {existing_VREs_data_path}")
else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")

# Boundary

- Prepare regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_regions_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by="Country")[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

In [ ]:
# from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
# from shapely.ops import unary_union

# def remove_holes(geom):
#     """Keep only exterior shells; drop all interior rings."""
#     if geom is None or geom.is_empty:
#         return geom

#     if geom.geom_type == "Polygon":
#         return Polygon(geom.exterior)

#     if geom.geom_type == "MultiPolygon":
#         parts = [Polygon(part.exterior) for part in geom.geoms if not part.is_empty]
#         return MultiPolygon(parts) if parts else geom

#     if geom.geom_type == "GeometryCollection":
#         polys = []
#         for g in geom.geoms:
#             cleaned = remove_holes(g)
#             if cleaned is not None and not cleaned.is_empty:
#                 if cleaned.geom_type in ["Polygon", "MultiPolygon"]:
#                     polys.append(cleaned)
#         return unary_union(polys) if polys else geom

#     return geom

In [ ]:
# all_boundary_clean= all_regions_boundary_dissolved_plot.copy()

# # fix possible invalid geometries first
# all_boundary_clean["geometry"] = all_boundary_clean.geometry.buffer(0)

# # remove interior holes
# all_boundary_clean["geometry"] = all_boundary_clean.geometry.apply(remove_holes)

# # optional: simplify outer boundary a bit
# all_boundary_clean["geometry"] = all_boundary_clean.geometry.simplify(
#     tolerance=5000,   # adjust based on CRS units
#     preserve_topology=True
# )

# all_boundary_clean.boundary.plot(
#     color="none",
#     edgecolor="#222222",
#     linewidth=0.7,
#     ax=ax,
#     zorder=3
# )

- Calculate Country Aggregated data

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

all_cells_aggr = get_sub_nationally_aggregated_capacity(all_cells_gdf, "Country")
aggregated_map = all_regions_boundary_dissolved_plot.merge(
    all_cells_aggr[
        [
            "Country",
            "potential_capacity_solar",
            "potential_capacity_wind",
            "Developable_area_solar",
            "Developable_area_wind",
            "geom_area_km2",
        ]
    ],
    on="Country",
    how="left",
)


- Add calculated data

In [ ]:
aggregated_map["potential_capacity_solar_GW"] = aggregated_map["potential_capacity_solar"] / 1e3
aggregated_map["potential_capacity_wind_GW"] = aggregated_map["potential_capacity_wind"] / 1e3
aggregated_map['Availability_solar_pct']=aggregated_map['Developable_area_solar']/aggregated_map['geom_area_km2']*100
aggregated_map['Availability_wind_pct']=aggregated_map['Developable_area_wind']/aggregated_map['geom_area_km2']*100

- plot

In [ ]:
if aggregated_map.crs != CRS_m:
    aggregated_map_plot = aggregated_map.to_crs(CRS_m)
else:
    aggregated_map_plot = aggregated_map

- Plot combined Availability

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib as mpl

# Convert to percent
all_cells_gdf_plot["LandAvailability_solar_pct"] = all_cells_gdf_plot["LandAvailability_ERA5_solar"] * 100
all_cells_gdf_plot["LandAvailability_wind_pct"] = all_cells_gdf_plot["LandAvailability_ERA5_wind"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=1000)
# fig.suptitle(
#     f"{country_name}: Land Availability for VRE development",
#     fontsize=16,
#     fontweight="bold",
#     y=0.99
# )

plot_specs = [
    ("LandAvailability_solar_pct", "Availability_solar_pct", "Solar", axes[0]),
    ("LandAvailability_wind_pct", "Availability_wind_pct", "Wind", axes[1]),
]

cmap = mpl.colormaps.get_cmap("Greens")
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# label offsets in map units
name_y_offset = 0
value_y_offset = 18000   # adjust if needed for your CRS/extent

for cell_col, agg_col, panel_title, ax in plot_specs:
    all_cells_gdf_plot.plot(
        column=cell_col,
        cmap=cmap,
        edgecolor="white",
        linewidth=0.3,
        legend=False,
        vmin=0,
        vmax=100,
        ax=ax,
    )

    aggregated_map_plot.plot(
        color="none",
        edgecolor="#222222",
        linewidth=0.7,
        ax=ax,
        zorder=3
    )

    for _, row in aggregated_map_plot.iterrows():
        point = row.geometry.representative_point()
        x, y = point.x, point.y

        # country name
        txt_name = ax.annotate(
            row["Country"],
            xy=(x, y + name_y_offset),
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="black",
            zorder=5
        )
        txt_name.set_path_effects([
            pe.withStroke(linewidth=2.5, foreground="white")
        ])

        # panel-specific availability value
        txt_val = ax.annotate(
            f"{row[agg_col]:.1f}%",
            xy=(x, y + value_y_offset),
            ha="center",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color="black",
            zorder=5
        )
        txt_val.set_path_effects([
            pe.withStroke(linewidth=3, foreground="lightyellow",alpha=0.8)
        ])

    ax.set_title(panel_title, fontsize=14, fontweight="bold")
    ax.set_axis_off()

# Layout
fig.subplots_adjust(bottom=0.1, wspace=0.08)

# Dedicated colorbar axis
cax = fig.add_axes([0.2, 0.02, 0.6, 0.025])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
cbar.set_label("Land availability (%)", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=12)
cbar.set_ticks([0, 20, 40, 60, 80, 100])

plt.savefig(
    f"{vis_save_to_root}/{country_name}_map_LandAvailability_solar_wind.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Attribute Maps

## Load Capacity and Scores

In [ ]:
if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:
    if existing_VREs_gdf.crs != CRS_m:
        existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
    else:
        existing_VREs_plot = existing_VREs_gdf

### Aggregated Capacity

- Calculate

In [ ]:
import matplotlib.patheffects as pe

# ========= Create subplots =========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=500)
Country_name_Y_adjustment:float=12E3 #in meters for CRS_m


# ========= Plot solar capacity =========
aggregated_map_plot.plot(
    column="potential_capacity_solar_GW",
    cmap="YlOrRd",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax1,
    legend_kwds={"label": "Solar Potential (GW)", "shrink": 0.7}
)
ax1.set_title("Solar Potential Capacity (GW)", fontsize=15, weight="bold")
ax1.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_solar_GW"]
    # Capacity value
    ax1.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# ========= Plot wind capacity ==============
aggregated_map_plot.plot(
    column="potential_capacity_wind_GW",
    cmap="BuPu",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax2,
    legend_kwds={"label": "Wind Potential (GW)", "shrink": 0.7}
)
ax2.set_title("Wind Potential Capacity (GW)", fontsize=15, weight="bold")
ax2.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    # Capacity value
    ax2.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.45, 0.88), ncol=2, fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(vis_save_to_root/f"{country_kwd}_Capacity_by_Country.png", bbox_inches='tight', transparent=False)

# Attribute's Map

### Capacity Factor

* Individual Maps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='CF',
                cell_edge_color='white',
                cell_linewidth=0.2,
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    target_crs=CRS_m,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    marker_scale_existing=15,
                                                    marker_highlight_width=2)

existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                     marker_scale_existing=14,
                                                    marker_highlight_width=2)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.8), ncol=1, fontsize=8, frameon=False)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m

# Plot only boundaries (no fill)
aggregated_map_plot.boundary.plot(ax=ax1, color='black', linewidth=0.7,alpha=0.7, zorder=3)
aggregated_map_plot.boundary.plot(ax=ax2, color='black', linewidth=0.7,alpha=0.7, zorder=3)

# Add text annotations for capacity and country name
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CF.png", bbox_inches='tight', transparent=False)

### Capacity

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)
# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended

# fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='solar',
                datafield='capacity',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(all_cells_gdf_plot, 
                resource_type='wind',
                datafield='capacity',
                ax=ax2, 
                show=False)

# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m

# Plot only boundaries (no fill)
aggregated_map_plot.boundary.plot(ax=ax1, color='#7A7A7A', linewidth=0.8,alpha=0.5, zorder=3)
aggregated_map_plot.boundary.plot(ax=ax2, color='k', linewidth=0.8, alpha=0.5,zorder=3)

# Add text annotations for capacity and country name
for idx, row in aggregated_map_plot.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CAPACITY.png", bbox_inches='tight', transparent=False)

## Score

#### Trimmed Maps Aggregated Capacity with LCOE thresholds 

##### Solar (<=80 USD/MWh), Wind (<=120USD/MWh), with Haircut
- __Capacity Haircut__ is downscaling total potential as a proxy of __permitting and acceptance constraints__.


- plot func

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe


def plot_resources_and_capacity_with_threshold_capacity_haircut(
    all_cells_gdf: gpd.GeoDataFrame,
    solar_lcoe_threshold: float | None=None,
    wind_lcoe_threshold: float | None=None,
    solar_capacity_haircut: float | None = None,
    wind_capacity_haircut: float | None = None,
    figsize: tuple = (12, 6),
    dpi: int = 600,
    save_path: str | None = None,
):
    """
    Plot solar and wind resource maps side by side with:
    - optional LCOE threshold filtering
    - optional capacity haircut factors
    - existing VRE overlays
    - country-level aggregated capacity annotations

    Assumes the following external objects/functions already exist:
    - CRS_m
    - vis.get_data_in_map_plot(...)
    - vis.get_existing_committed_VRE_plot(...)
    - get_sub_nationally_aggregated_capacity(...)
    - existing_VREs_plot
    - all_regions_boundary_dissolved
    - utils.print_warning(...)
    """

    # -------------------------------------------------
    # Validate required columns
    # -------------------------------------------------
    required_cols = {
        "Country",
        "geometry",
        "lcoe_solar",
        "lcoe_wind",
        "potential_capacity_solar",
        "potential_capacity_wind",
    }
    missing = required_cols - set(all_cells_gdf.columns)
    if missing:
        raise ValueError(f"all_cells_gdf is missing required columns: {sorted(missing)}")

    if all_cells_gdf.empty:
        raise ValueError("all_cells_gdf is empty.")

    # -------------------------------------------------
    # CRS harmonization
    # -------------------------------------------------
    if all_cells_gdf.crs != CRS_m:
        all_cells_plot = all_cells_gdf.to_crs(CRS_m)
    else:
        all_cells_plot = all_cells_gdf.copy()

    if all_regions_boundary_dissolved.crs != CRS_m:
        boundaries_plot = all_regions_boundary_dissolved.to_crs(CRS_m)
    else:
        boundaries_plot = all_regions_boundary_dissolved.copy()

    if existing_VREs_plot.crs != CRS_m:
        existing_vres_plot = existing_VREs_plot.to_crs(CRS_m)
    else:
        existing_vres_plot = existing_VREs_plot.copy()

    # -------------------------------------------------
    # Default haircut = 1.0 if not provided
    # -------------------------------------------------
    solar_capacity_haircut = 1.0 if solar_capacity_haircut is None else solar_capacity_haircut
    wind_capacity_haircut = 1.0 if wind_capacity_haircut is None else wind_capacity_haircut

    # Optional validation
    if solar_capacity_haircut < 0 or wind_capacity_haircut < 0:
        raise ValueError("Capacity haircut factors must be >= 0.")

    # -------------------------------------------------
    # Threshold filtering
    # -------------------------------------------------
    if solar_lcoe_threshold is not None:
        cells_thresholded_solar = all_cells_plot.loc[
            all_cells_plot["lcoe_solar"] <= solar_lcoe_threshold
        ].copy()
    else:
        utils.print_warning("No solar LCOE threshold provided; plotting all cells for solar.")
        cells_thresholded_solar = all_cells_plot.copy()

    if wind_lcoe_threshold is not None:
        cells_thresholded_wind = all_cells_plot.loc[
            all_cells_plot["lcoe_wind"] <= wind_lcoe_threshold
        ].copy()
    else:
        utils.print_warning("No wind LCOE threshold provided; plotting all cells for wind.")
        cells_thresholded_wind = all_cells_plot.copy()

    # -------------------------------------------------
    # Create figure
    # -------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, dpi=dpi)

    # -------------------------------------------------
    # Gray base first: all cells
    # This becomes the "excluded / not selected" background
    # -------------------------------------------------
    all_cells_plot.plot(ax=ax1, color="grey", edgecolor="none", alpha=0.7, zorder=1)
    all_cells_plot.plot(ax=ax2, color="grey", edgecolor="none", alpha=0.7, zorder=1)

    # -------------------------------------------------
    # Resource maps on top
    # -------------------------------------------------
    if not cells_thresholded_solar.empty:
        vis.get_data_in_map_plot(
            cells_thresholded_solar,
            resource_type="solar",
            datafield="score",
            compass_size=12,
            ax=ax1,
            score_threshold=250,
            show=False,
        )

    if not cells_thresholded_wind.empty:
        vis.get_data_in_map_plot(
            cells_thresholded_wind,
            resource_type="wind",
            datafield="score",
            compass_size=12,
            ax=ax2,
            score_threshold=250,
            show=False,
        )

    no_land_patch = mpatches.Patch(
        facecolor="lightgray",
        edgecolor="gray",
        alpha=0.7,
        label="Economically unfeasible or no developable land",
    )

    # -------------------------------------------------
    # Existing solar / wind overlays
    # -------------------------------------------------
    solar_legends = []
    wind_legends = []

    if "Technology" not in existing_vres_plot.columns:
        raise ValueError("existing_VREs_plot must contain a 'Technology' column.")

    existing_vres_solar = existing_vres_plot.loc[
        existing_vres_plot["Technology"].astype(str).str.lower() == "solar"
    ].copy()

    existing_vres_wind = existing_vres_plot.loc[
        existing_vres_plot["Technology"].astype(str).str.lower() == "wind"
    ].copy()

    if not existing_vres_solar.empty:
        ax1, solar_legends = vis.get_existing_committed_VRE_plot(
            ax=ax1,
            existing_VREs_gdf=existing_vres_solar,
            existing_VRE_type_column="Technology",
            target_crs=CRS_m,
            marker_scale_existing=14,
            marker_highlight_width=3,
        )

    if not existing_vres_wind.empty:
        ax2, wind_legends = vis.get_existing_committed_VRE_plot(
            ax=ax2,
            existing_VREs_gdf=existing_vres_wind,
            existing_VRE_type_column="Technology",
            target_crs=CRS_m,
            marker_scale_existing=14,
            marker_highlight_width=3,
        )

    # -------------------------------------------------
    # Aggregated capacity using thresholded cells
    # IMPORTANT: use filtered subsets, not all cells
    # -------------------------------------------------
    solar_aggr = get_sub_nationally_aggregated_capacity(cells_thresholded_solar, "Country").copy()
    wind_aggr = get_sub_nationally_aggregated_capacity(cells_thresholded_wind, "Country").copy()

    # Convert MW -> GW if your source is MW
    solar_aggr["potential_capacity_solar_GW_threshold"] = (
        solar_aggr["potential_capacity_solar"] * solar_capacity_haircut / 1e3
    )
    wind_aggr["potential_capacity_wind_GW_threshold"] = (
        wind_aggr["potential_capacity_wind"] * wind_capacity_haircut / 1e3
    )

    # Merge to boundary map for geometry consistency
    solar_map = boundaries_plot.merge(
        solar_aggr[["Country", "potential_capacity_solar_GW_threshold"]],
        on="Country",
        how="left",
    )
    wind_map = boundaries_plot.merge(
        wind_aggr[["Country", "potential_capacity_wind_GW_threshold"]],
        on="Country",
        how="left",
    )

    # Country boundaries
    solar_map.boundary.plot(ax=ax1, color="black", linewidth=0.7,alpha=0.7, zorder=6)
    wind_map.boundary.plot(ax=ax2, color="black", linewidth=0.7,alpha=0.7, zorder=6)

    # -------------------------------------------------
    # Country labels + capacities
    # representative_point() is safer than centroid
    # -------------------------------------------------
    Country_name_Y_adjustment = 12_000

    for _, row in solar_map.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        cap = row.get("potential_capacity_solar_GW_threshold")
        if cap is None:
            continue

        label_point = row.geometry.representative_point()
        x, y = label_point.x, label_point.y

        ax1.annotate(
            f"{cap:.1f}",
            (x, y),
            color="black",
            fontsize=10,
            ha="center",
            va="center",
            fontweight="bold",
            zorder=7,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
        )
        ax1.annotate(
            row["Country"],
            (x, y + Country_name_Y_adjustment),
            color="black",
            fontsize=9,
            ha="center",
            va="bottom",
            zorder=7,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
        )

    for _, row in wind_map.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        cap = row.get("potential_capacity_wind_GW_threshold")
        if cap is None:
            continue

        label_point = row.geometry.representative_point()
        x, y = label_point.x, label_point.y

        ax2.annotate(
            f"{cap:.1f}",
            (x, y),
            color="black",
            fontsize=10,
            ha="center",
            va="center",
            fontweight="bold",
            zorder=7,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
        )
        ax2.annotate(
            row["Country"],
            (x, y + Country_name_Y_adjustment),
            color="black",
            fontsize=9,
            ha="center",
            va="bottom",
            zorder=7,
            path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.7)],
        )

    # -------------------------------------------------
    # Axes cleanup
    # -------------------------------------------------
    ax1.set_axis_off()
    ax2.set_axis_off()

    # -------------------------------------------------
    # Shared legend
    # -------------------------------------------------
    all_legends = []
    all_legends.extend(solar_legends if solar_legends is not None else [])
    all_legends.extend(wind_legends if wind_legends is not None else [])
    all_legends.append(no_land_patch)

    # de-duplicate legend labels
    unique_handles = []
    seen_labels = set()
    for h in all_legends:
        label = h.get_label()
        if label not in seen_labels:
            unique_handles.append(h)
            seen_labels.add(label)

    fig.legend(
        handles=unique_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.90),
        ncol=1,
        fontsize=9.5,
        frameon=False,
    )

    # -------------------------------------------------
    # Plot note
    # -------------------------------------------------
    note_parts = []

    if solar_lcoe_threshold is not None:
        note_parts.append(f"Solar LCOE threshold ≤ {solar_lcoe_threshold:.1f} \\$/MWh")
    else:
        note_parts.append("Solar LCOE threshold: not applied")

    if wind_lcoe_threshold is not None:
        note_parts.append(f"Wind LCOE threshold ≤ {wind_lcoe_threshold:.1f} \\$/MWh")
    else:
        note_parts.append("Wind LCOE threshold: not applied")

    # only show haircut text if explicitly different from 1.0
    haircut_parts = []
    if solar_capacity_haircut != 1.0:
        haircut_parts.append(f"solar = {solar_capacity_haircut:.2f}")
    if wind_capacity_haircut != 1.0:
        haircut_parts.append(f"wind = {wind_capacity_haircut:.2f}")

    if haircut_parts:
        note_parts.append("Capacity haircut applied: " + "; ".join(haircut_parts))

    plot_note = " | ".join(note_parts)

    fig.text(
        0.5,
        0.02,
        plot_note,
        ha="center",
        va="bottom",
        fontsize=9.5,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="gray",
            alpha=0.9,
        ),
    )

    # -------------------------------------------------
    # Layout and save
    # -------------------------------------------------
    plt.tight_layout(rect=[0, 0.05, 1, 0.92])

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", transparent=False)

    plt.show()

    return fig, (ax1, ax2)

- Define _lcoe_ thresholds and capacity haircuts

In [ ]:
solar_lcoe_threshold=70 
wind_lcoe_threshold=130
solar_capacity_haircut=0.8
wind_capacity_haircut=0.6

# Interactive Map

In [ ]:
m = vis.make_lcoe_map(
    wind_gdf=all_cells_gdf,
    solar_gdf=all_cells_gdf,
    sub_national_unit_tag=sub_national_unit_tag,
    save_path=vis_save_to_root / f"{country_kwd}_lcoe_map_{RUN_ID}.html",
    basemap_tiles="Esri WorldGrayCanvas",
    wind_lcoe_max=wind_lcoe_threshold,
    solar_lcoe_max=solar_lcoe_threshold,
)

In [ ]:
vis_save_to_root / f"{country_kwd}_lcoe_map_{RUN_ID}.html"

- plot thresholded map without haircut

In [ ]:
# =========================================================
# Plot WITHOUT lcoe threshold and haircut
# =========================================================
plot_resources_and_capacity_with_threshold_capacity_haircut(
    all_cells_gdf=all_cells_gdf,
    save_path=vis_save_to_root / "Resources_and_Capacity_Combined_with_all_cells.png",
    # title="Resources and Capacity by Country",
    figsize=(12,6),
    dpi=500,
)
# =========================================================
# Plot with threshold but WITHOUT haircut
# =========================================================
plot_resources_and_capacity_with_threshold_capacity_haircut(
    all_cells_gdf=all_cells_gdf,
    solar_lcoe_threshold=solar_lcoe_threshold,
    wind_lcoe_threshold=wind_lcoe_threshold,
    save_path=vis_save_to_root / "Resources_and_Capacity_Combined_with_LCOE_Thresholds.png",
    # title="Resources and Capacity by Country",
    solar_capacity_haircut=1,
    wind_capacity_haircut=1,
    figsize=(12,6),
    dpi=500,
)

# # =========================================================
# # Plot WITH haircut
# # =========================================================
plot_resources_and_capacity_with_threshold_capacity_haircut(
    all_cells_gdf=all_cells_gdf,
    solar_lcoe_threshold=solar_lcoe_threshold,
    wind_lcoe_threshold=wind_lcoe_threshold,
    save_path=vis_save_to_root / "Resources_and_Capacity_Combined_with_LCOE_Thresholds_CapacityHaircut.png",
    # title="Resources and Capacity by Country",
    solar_capacity_haircut=solar_capacity_haircut,
    wind_capacity_haircut=wind_capacity_haircut,
    figsize=(12,6),
    dpi=500,
)

## Clustering

### Cells

- Method 1 (simple)

In [ ]:
def get_country_clusters_cell(all_cells_df: pd.DataFrame, 
                              solar_lcoe_threshold: float,
                              wind_lcoe_threshold: float):
    """
    Get country-level clusters of cells that meet the LCOE thresholds for solar and wind.
    Aggregates potential capacity, area, and costs for each resource by country.
    Returns two DataFrames: one for solar and one for wind, each with aggregated metrics.
    """
    assert 'Country' in all_cells_df.columns, "Input DataFrame must contain 'Country' column."
    assert 'lcoe_solar' in all_cells_df.columns, "Input DataFrame must contain 'lcoe_solar' column."
    assert 'lcoe_wind' in all_cells_df.columns, "Input DataFrame must contain 'lcoe_wind' column."
    
    for resources in ['solar','wind']:
        if resources == 'solar':
            cells = all_cells_df[all_cells_df['lcoe_solar']<=solar_lcoe_threshold].copy()
        else:
            cells = all_cells_df[all_cells_df['lcoe_wind']<=wind_lcoe_threshold].copy()
        

        cells["Country"] = cells["Country"].apply(utils.standardize_tags)
        
        agg_dict = {
            f"potential_capacity_{resources}": "sum",
            "geom_area_km2": "sum",
            f"capex_{resources}": "sum",
            f"fom_{resources}": "sum",
            f"vom_{resources}": "mean",
            f"Developable_area_{resources}":"sum",
            f"grid_connection_cost_per_km_{resources}": "mean",
            f"tx_line_rebuild_cost_{resources}": "mean",
            f"Operational_life_{resources}": "mean",
            f"{resources}_CF_mean": "mean",
            f"lcoe_{resources}": "mean",
        }

        if resources == 'solar':
            solar_clusters = cells.groupby("Country").agg(agg_dict).reset_index()
        else:
            wind_clusters = cells.groupby("Country").agg(agg_dict).reset_index()
    return solar_clusters, wind_clusters

- Method 2 (Standardized to absorb other clustering methods)

In [ ]:
# from RES import cluster
# for resources in ['solar','wind']:
#     if resources == 'solar':
#         cells = solar_cells
#     else:
#         cells = wind_cells
#     cells["Country"] = cells["Country"].apply(utils.standardize_tags)
#     dataframe,optimal_k_df=cluster.pre_process_cluster_mapping(cells,'cluster_temp',1,'Country',resources)
#     optimal_k_df["Country"] = optimal_k_df["Country"].apply(utils.standardize_tags)
#     dataframe_filtered=dataframe[dataframe['Country'].isin(list(optimal_k_df['Country']))]
#     cluster_map,region_optimal_k_df = cluster.cells_to_cluster_mapping(cells_scored=cells, 
#                                                                    vis_directory='cluster_temp', 
#                                                                    wcss_tolerance=1,
#                                                                    sub_national_unit_tag='Country',
#                                                                    resource_type=resources,
#                                                                    sort_columns=[f'lcoe_{resources}', f'potential_capacity_{resources}']
#                                                                    )
#     cluster_map['Country'] = cluster_map['Country'].apply(utils.standardize_tags)
#     region_optimal_k_df['Country'] = region_optimal_k_df['Country'].apply(utils.standardize_tags)
#     cell_cluster_gdf, dissolved_indices = cluster.create_cells_Union_in_clusters(cluster_map, 
#                                                                                             region_optimal_k_df,
#                                                                                             'Country',
#                                                                                             resources)
#     cell_cluster_gdf['potential_capacity_GW'] = cell_cluster_gdf['potential_capacity']/1E3
#     if resources == 'solar':
#         solar_dissolved_indices=dissolved_indices
#         solar_clusters=cell_cluster_gdf#[['lcoe', 'potential_capacity_GW', 'CF_mean','Country','Cluster_no'] ]
#     else:
#         wind_dissolved_indices=dissolved_indices
#         wind_clusters=cell_cluster_gdf#[['lcoe', 'potential_capacity_GW', 'CF_mean','Country','Cluster_no'] ]

In [ ]:
solar_clusters, wind_clusters = get_country_clusters_cell(all_cells_gdf, 
                                                           solar_lcoe_threshold=solar_lcoe_threshold, 
                                                           wind_lcoe_threshold=wind_lcoe_threshold)

In [ ]:
country_name_to_code = {
    v["name"].replace(" ", ""): k
    for k, v in cfg.get("region_mapping").items()
}

# Map Country -> Country_code
solar_clusters["Country_code"] = solar_clusters["Country"].map(country_name_to_code)
solar_clusters=solar_clusters.set_index(solar_clusters["Country_code"])
solar_clusters.to_csv(results_save_to_root/"solar_clusters.csv", index=False)
wind_clusters["Country_code"] = wind_clusters["Country"].map(country_name_to_code)
wind_clusters=wind_clusters.set_index(wind_clusters["Country_code"])
wind_clusters.to_csv(results_save_to_root/"wind_clusters.csv", index=False)

In [ ]:
wind_clusters

In [ ]:
solar_clusters

### Timeseries

In [ ]:
solar_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_solar'] for region in combined_store]
wind_ts_dict:dict[pd.DataFrame]=[combined_store[region]['timeseries_wind'] for region in combined_store]

In [ ]:
solar_ts=pd.DataFrame()
wind_ts=pd.DataFrame()
for resource_type in ['solar','wind']:
    for region in combined_store.keys():
        ts_df=combined_store[region][f'timeseries_{resource_type}']
        ts_mean=ts_df.mean(axis=1) # to check if there is any non-zero value, if all values are zero then it means timeseries data is not available for that region
        ts_mean['Country']=region
        ts_mean.index.name="timestamp"
        if resource_type == 'solar':
            solar_ts[region]=ts_mean

        else:
            wind_ts[region]=ts_mean
            
            
wind_ts.to_csv(results_save_to_root/f"wind_timeseries.csv", index=True)
print(f"Saved wind timeseries to {results_save_to_root/f'wind_timeseries.csv'}")
solar_ts.to_csv(results_save_to_root/f"solar_timeseries.csv", index=True)
print(f"Saved solar timeseries to {results_save_to_root/f'solar_timeseries.csv'}")

In [ ]:
# wind_ts

#### plot

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

def plot_ts_dashboard(ts_df: pd.DataFrame, resource: str = "solar"):
    """
    Interactive dashboard for hourly CF timeseries by country.

    Parameters:
        ts_df: DataFrame with datetime index and country columns (e.g. AL, BA, ...).
        resource: Label for titles ("solar" or "wind").
    """
    # Drop any non-numeric rows (e.g. the 'Country' label row at the bottom)
    ts_df = ts_df.apply(pd.to_numeric, errors="coerce").dropna()
    ts_df.index = pd.to_datetime(ts_df.index)

    daily_mean = ts_df.resample("D").mean()
    countries = ts_df.columns.tolist()

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            f"Hourly {resource.title()} CF — All Countries",
            f"Daily Mean {resource.title()} CF — All Countries",
            f"Mean Diurnal Profile (Hour of Day)",
        ),
        vertical_spacing=0.08,
        row_heights=[0.4, 0.35, 0.25],
    )

    # --- Row 1: Hourly profiles ---
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=ts_df.index,
                y=ts_df[country],
                name=country,
                legendgroup=country,
                mode="lines",
                opacity=0.7,
                line=dict(width=0.8),
            ),
            row=1, col=1,
        )

    # --- Row 2: Daily mean ---
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=daily_mean.index,
                y=daily_mean[country],
                name=country,
                legendgroup=country,
                showlegend=False,
                mode="lines",
                line=dict(width=1.2),
            ),
            row=2, col=1,
        )

    # --- Row 3: Mean diurnal profile ---
    diurnal = ts_df.groupby(ts_df.index.hour).mean()
    for country in countries:
        fig.add_trace(
            go.Scatter(
                x=diurnal.index,
                y=diurnal[country],
                name=country,
                legendgroup=country,
                showlegend=False,
                mode="lines+markers",
                line=dict(width=2),
            ),
            row=3, col=1,
        )

    fig.update_xaxes(title_text="Time", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_xaxes(title_text="Hour of Day", dtick=1, row=3, col=1)

    fig.update_yaxes(title_text="CF", row=1, col=1)
    fig.update_yaxes(title_text="CF (daily mean)", row=2, col=1)
    fig.update_yaxes(title_text="CF (mean)", row=3, col=1)

    fig.update_layout(
        height=1000,
        title_text=f"{resource.title()} Capacity Factor Dashboard",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        hovermode="x unified",
    )

    fig.show()
    fig.write_html(vis_save_to_root/f"{resource}_CF_dashboard.html")


# --- Usage ---
# plot_ts_dashboard(wind_ts_df, resource="wind")
# plot_ts_dashboard(solar_ts_df, resource="solar")

In [ ]:
plot_ts_dashboard(wind_ts, resource="wind")
plot_ts_dashboard(solar_ts, resource="solar")